In [1]:
from IPython.display import HTML

with open('../frontend/dashboard.html', 'r', encoding='utf-8') as f:
    html = f.read()

if not html.strip().endswith('</html>'):
    html += '\n</html>'

extra_css = """
<style>
.topbar {
    position: sticky !important;
    top: 0 !important;
    z-index: 100 !important;
}
.sidebar {
    position: sticky !important;
    top: 56px !important;
    height: calc(100vh - 56px) !important;
    overflow-y: auto !important;
    z-index: 90 !important;
}
.form-page {
    position: absolute !important;
    top: 0 !important;
    left: 0 !important;
    right: 0 !important;
    min-height: 100% !important;
    z-index: 1000 !important;
    background: var(--white) !important;
    overflow-y: auto !important;
}
body {
    position: relative !important;
}
.layout {
    padding-top: 0 !important;
}
</style>
"""

fix_js = """
<script>
// Fix modal open/close
window.closeModal = function(id) {
    const modal = document.getElementById(id);
    if (modal) {
        modal.classList.remove('open');
        modal.style.display = 'none';
        document.body.style.overflow = '';
    }
};
window.closeFormPage = window.closeModal;

window.openModal = function(id) {
    const modal = document.getElementById(id);
    if (modal) {
        modal.style.display = 'block';
        modal.classList.add('open');
        document.body.style.overflow = 'hidden';
    }
};

// Fix CSV import - correct parsing van alle velden
window.importCSV = function(type) {
    const input = document.getElementById('csv-input');
    input.onchange = (e) => {
        const file = e.target.files[0];
        if (!file) return;
        const reader = new FileReader();
        reader.onload = (ev) => {
            const lines = ev.target.result.trim().split('\\n');
            const headers = lines[0].split(',').map(h => h.trim());
            const dataLines = lines.slice(1);

            if (type === 'mw') {
                dataLines.forEach(line => {
                    if (!line.trim()) return;
                    const values = line.split(',').map(v => v.trim());
                    const row = {};
                    headers.forEach((h, i) => row[h] = values[i] || '');

                    const naam = row['Naam'] || row['naam'] || values[0];
                    if (!naam) return;

                    state.medewerkers.push({
                        id: Date.now() + Math.random(),
                        naam: naam,
                        email: row['Email'] || row['email'] || '',
                        telefoon: row['Telefoon'] || row['telefoon'] || '',
                        straat: row['Straat'] || row['straat'] || '',
                        postcode: row['Postcode'] || row['postcode'] || '',
                        stad: row['Stad'] || row['stad'] || '',
                        startTijd: row['Start Tijd'] || row['start tijd'] || '08:00',
                        eindTijd: row['Eind Tijd'] || row['eind tijd'] || '17:00',
                        dagen: (row['Dagen'] || row['dagen'] || '').split(';').map(d => d.trim()).filter(Boolean),
                        color: ['blue','green','purple','orange'][state.medewerkers.length % 4]
                    });
                });
                renderMedewerkers();

            } else {
                dataLines.forEach(line => {
                    if (!line.trim()) return;
                    const values = line.split(',').map(v => v.trim());
                    const row = {};
                    headers.forEach((h, i) => row[h] = values[i] || '');

                    const naam = row['Naam'] || row['naam'] || values[0];
                    if (!naam) return;

                    state.clienten.push({
                        id: Date.now() + Math.random(),
                        naam: naam,
                        telefoon: row['Telefoon'] || row['telefoon'] || '',
                        typeZorg: row['Type Zorg'] || row['type zorg'] || '',
                        duur: parseInt(row['Duur (min)'] || row['duur'] || '60') || 60,
                        straat: row['Straat'] || row['straat'] || '',
                        postcode: row['Postcode'] || row['postcode'] || '',
                        stad: row['Stad'] || row['stad'] || '',
                        dagen: (row['Dagen'] || row['dagen'] || '').split(';').map(d => d.trim()).filter(Boolean),
                        tijdvensters: row['Tijdvensters'] || row['tijdvensters'] || '',
                        opmerkingen: row['Opmerkingen'] || row['opmerkingen'] || ''
                    });
                });
                renderClienten();
            }

            updateStats();
            toast('CSV geïmporteerd (' + (type === 'mw' ? state.medewerkers.length : state.clienten.length) + ' records)', 'success');
        };
        reader.readAsText(file);
    };
    input.value = '';
    input.click();
};
</script>
"""

html = html.replace('<style>', extra_css + '<style>', 1)
html = html.replace('</body>', fix_js + '\n</body>')

HTML(html)